# Notebook 1: Returns and Episodes

Before we touch any RL algorithm, we need to answer a simple question: **how do you measure how good a strategy is?**

This notebook builds the foundations. By the end you'll understand:
- What episodes and returns are
- Why we discount future rewards
- How gamma shapes an agent's behavior
- Why naive return computation fails and what to do about it

In [1]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

## Part 1: A World With Tradeoffs

Here's a simple game. You're on a 1D grid with 7 positions (0-6). You start at position 2.

```
  [+3]  .  @  .  .  .  [+8]
   0    1  2  3  4  5    6
```

- **Position 0**: Small treasure worth +3 (2 steps left)
- **Position 6**: Big treasure worth +8 (4 steps right)
- **Every step**: Costs -1.0 (traveling is hard)
- **Timeout at 15 steps**: -3 additional penalty

Each step you choose: go left (0) or go right (1). The episode ends when you reach a treasure or time out.

**The question: which treasure should you go for?**

The big treasure is worth more than double — but it's also twice as far. Every extra step costs you. Is it worth the journey?

In [2]:
class TreasureHunter:
    """A simple 1D environment with a genuine strategic tradeoff."""
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.pos = 2
        self.tick = 0
        self.done = False
        return self.pos
    
    def step(self, action):
        """action: 0 = left, 1 = right. Returns (pos, reward, done)"""
        assert not self.done, "Episode is over, call reset()"
        assert action in (0, 1), "Action must be 0 (left) or 1 (right)"
        
        self.pos += -1 if action == 0 else 1
        self.pos = max(0, min(6, self.pos))  # clamp to grid
        self.tick += 1
        
        reward = -1.0  # step cost
        
        if self.pos == 0:
            reward += 3.0  # small treasure
            self.done = True
        elif self.pos == 6:
            reward += 8.0  # big treasure
            self.done = True
        elif self.tick >= 15:
            reward += -3.0  # timeout penalty
            self.done = True
        
        return self.pos, reward, self.done

Let's play two episodes manually and record what happens.

In [3]:
env = TreasureHunter()

# Strategy: always go LEFT
env.reset()
left_rewards = []
while not env.done:
    pos, reward, done = env.step(0)
    left_rewards.append(reward)
    
print(f"Go left:  rewards = {left_rewards}")
print(f"          total   = {sum(left_rewards):.1f}")
print(f"          steps   = {len(left_rewards)}")
print()

# Strategy: always go RIGHT
env.reset()
right_rewards = []
while not env.done:
    pos, reward, done = env.step(1)
    right_rewards.append(reward)

print(f"Go right: rewards = {right_rewards}")
print(f"          total   = {sum(right_rewards):.1f}")
print(f"          steps   = {len(right_rewards)}")

Go left:  rewards = [-1.0, 2.0]
          total   = 1.0
          steps   = 2

Go right: rewards = [-1.0, -1.0, -1.0, 7.0]
          total   = 4.0
          steps   = 4


## Part 2: The Return

The **return** of an episode is how we measure its total outcome. The simplest version: just add up all the rewards.

$$G = r_0 + r_1 + r_2 + \ldots + r_T$$

Run the cell above to see what going left vs right gives you. The total rewards are different — but by how much?

The return tells us the outcome of the **whole episode**. But what if we want to know the return **from a specific step onward**? This is called the return from timestep $t$:

$$G_t = r_t + r_{t+1} + r_{t+2} + \ldots + r_T$$

This matters because the agent needs to make a decision at **each** step, and it needs to know: "from this point forward, how much reward can I expect?"

### Exercise 1: Compute returns at each timestep

Given a list of rewards from an episode, compute the return at each timestep. `returns[t]` should be the sum of all rewards from step `t` onward.

**Think about this before coding:** can you compute this efficiently by working backward from the end?

In [ ]:
def compute_returns(rewards):
    """Compute the return at each timestep.
    
    Args:
        rewards: list of floats, reward at each timestep
    Returns:
        list of floats, return from each timestep onward
    
    Example: rewards = [1, 2, 3] -> returns = [6, 5, 3]
        returns[0] = 1 + 2 + 3 = 6
        returns[1] = 2 + 3 = 5
        returns[2] = 3
    """
    rewards_from_t = np.zeros(len(rewards))
    for i in range(len(rewards)):
        for j, reward in enumerate(rewards):
            if i > j:
                continue
            rewards_from_t[i] += reward
    print(rewards_from_t)
    return rewards_from_t


                


# --- Validation ---
assert compute_returns([1, 2, 3]) == [6, 5, 3], "Basic test failed"
assert compute_returns([0, 0, 0, 1]) == [1, 1, 1, 1], "Sparse reward test failed"
assert compute_returns([-1]) == [-1], "Single step test failed"
assert compute_returns([]) == [], "Empty test failed"

# Now test on the actual episodes
left_returns = compute_returns(left_rewards)
right_returns = compute_returns(right_rewards)
print(f"Go left:  returns at each step = {[f'{r:.1f}' for r in left_returns]}")
print(f"Go right: returns at each step = {[f'{r:.1f}' for r in right_returns]}")
print("\nPassed!")

[1. 3. 6.]


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

## Part 3: Why Discount?

The undiscounted return says going right (+4.0) is better than going left (+1.0). Case closed?

Not so fast. Consider these scenarios:

1. **Uncertainty**: What if there's a 20% chance per step that wind pushes you the wrong way? The longer you travel, the more likely something goes wrong.

2. **Infinite games**: What if the game never ends? The return would be infinite — useless for comparison.

3. **Present vs future**: A reward now is more certain than a reward later. Shouldn't we account for that?

The solution: **discounted returns**. We multiply each future reward by $\gamma^k$ where $k$ is how many steps in the future it is:

$$G_t = r_t + \gamma \cdot r_{t+1} + \gamma^2 \cdot r_{t+2} + \ldots$$

Or recursively (this form is important — it's how we'll compute it efficiently):

$$G_t = r_t + \gamma \cdot G_{t+1}$$

With $\gamma = 0.99$, a reward 100 steps away is worth $0.99^{100} \approx 0.37$ of its face value. With $\gamma = 0.5$, that same reward is worth $0.5^{100} \approx 0$ — essentially nothing.

### Exercise 2: Implement discounted returns

Modify your return computation to include discounting. Use the recursive form — it's one line of change from your previous solution.

**Hint:** Work backward. At the last step, $G_T = r_T$. At step $T-1$, $G_{T-1} = r_{T-1} + \gamma \cdot G_T$.

In [ ]:
def compute_discounted_returns(rewards, gamma):
    """Compute discounted return at each timestep.
    
    Args:
        rewards: list of floats
        gamma: float, discount factor (0 to 1)
    Returns:
        list of floats, discounted return from each timestep
    
    Example: rewards = [1, 2, 3], gamma = 0.5
        returns[2] = 3
        returns[1] = 2 + 0.5 * 3 = 3.5
        returns[0] = 1 + 0.5 * 3.5 = 2.75
    """
    # YOUR CODE HERE
    pass

# --- Validation ---
# With gamma=1.0, should be identical to undiscounted
assert compute_discounted_returns([1, 2, 3], 1.0) == [6, 5, 3], "gamma=1.0 should equal undiscounted"

# With gamma=0.0, each step only sees its own reward
assert compute_discounted_returns([1, 2, 3], 0.0) == [1, 2, 3], "gamma=0.0: only immediate reward"

# Manual check
result = compute_discounted_returns([1, 2, 3], 0.5)
assert abs(result[0] - 2.75) < 1e-6, f"Expected 2.75, got {result[0]}"
assert abs(result[1] - 3.5) < 1e-6, f"Expected 3.5, got {result[1]}"
assert abs(result[2] - 3.0) < 1e-6, f"Expected 3.0, got {result[2]}"

print("Passed!")

### Exercise 3: The Crossover

Now the interesting question. We have two strategies:
- **Go left**: 2 steps, net return = +1.0 (undiscounted)
- **Go right**: 4 steps, net return = +4.0 (undiscounted)

Going right is better undiscounted. But with discounting, the step costs (-1.0 each) hit harder the more steps you take. At some gamma, the quick small treasure becomes more attractive than the slow big one.

**Your task**: Find the approximate gamma value where the two strategies produce the same discounted return. Sweep gamma from 0 to 1, compute both returns, and plot them.

**Before you code: predict.** Do you think the crossover gamma is closer to 0.3, 0.6, or 0.9?

In [ ]:
def run_strategy(env, strategy):
    """Run one episode with a given strategy.
    strategy: 'left' or 'right'
    Returns: list of rewards
    """
    env.reset()
    rewards = []
    while not env.done:
        action = 0 if strategy == 'left' else 1
        _, reward, _ = env.step(action)
        rewards.append(reward)
    return rewards

env = TreasureHunter()
left_rewards = run_strategy(env, 'left')
right_rewards = run_strategy(env, 'right')

# YOUR CODE HERE
# 1. Create a range of gamma values from 0.0 to 1.0
# 2. For each gamma, compute the discounted return (at t=0) for both strategies
# 3. Plot both curves on the same graph
# 4. Find the approximate crossover gamma
#
# gammas = ...
# left_returns = ...
# right_returns = ...
# plt.plot(...)
# crossover_gamma = ...

# --- Validation ---
# Uncomment after implementing:
# assert 0.1 < crossover_gamma < 0.99, f"Crossover gamma {crossover_gamma} seems wrong"
# print(f"Crossover gamma ≈ {crossover_gamma:.3f}")
# print(f"Below this: go left. Above this: go right.")

## Part 4: Returns Are Noisy

So far our environment is deterministic — same strategy always gives the same return. Real environments are stochastic. Let's add some randomness and see what happens to our returns.

In [ ]:
class NoisyTreasureHunter(TreasureHunter):
    """Same game but with wind. Each step, 20% chance of being
    pushed in a random direction instead of your chosen one."""
    def step(self, action):
        if np.random.random() < 0.2:
            action = np.random.randint(0, 2)  # wind!
        return super().step(action)

### Exercise 4: Return distributions

Run both strategies 2000 times each on the noisy environment with gamma=0.95. Collect the discounted return (from t=0) for each episode.

Then:
1. Plot histograms of returns for both strategies (on the same plot)
2. Compute the mean and standard deviation for each
3. Answer: Is "go right" still the better strategy? How confident are you?

**Think about this before looking at results:** With 20% wind, how does the longer journey (go right) compare to the shorter one? What kinds of bad outcomes can happen?

In [ ]:
gamma = 0.95
n_episodes = 2000
env = NoisyTreasureHunter()

# YOUR CODE HERE
# 1. Run 2000 episodes for each strategy on the noisy env
# 2. For each episode, compute the discounted return at t=0
# 3. Store returns in left_returns_list and right_returns_list
#
# left_returns_list = []
# right_returns_list = []
# for _ in range(n_episodes):
#     ...

# --- Validation & Visualization ---
# Uncomment after implementing:
#
# left_mean = np.mean(left_returns_list)
# left_std = np.std(left_returns_list)
# right_mean = np.mean(right_returns_list)
# right_std = np.std(right_returns_list)
# 
# print(f"Go left:  mean = {left_mean:.2f}, std = {left_std:.2f}")
# print(f"Go right: mean = {right_mean:.2f}, std = {right_std:.2f}")
#
# plt.figure(figsize=(10, 5))
# plt.hist(left_returns_list, bins=30, alpha=0.6, label=f'Left (mean={left_mean:.2f})', color='blue')
# plt.hist(right_returns_list, bins=30, alpha=0.6, label=f'Right (mean={right_mean:.2f})', color='red')
# plt.xlabel('Discounted Return')
# plt.ylabel('Count')
# plt.title('Return Distributions (gamma=0.95, 20% wind)')
# plt.legend()
# plt.show()
#
# assert len(left_returns_list) == n_episodes
# assert len(right_returns_list) == n_episodes
# print("Passed!")

### Exercise 5: Risk and Reward

The histograms show something important: the "go right" strategy has **higher variance**. Some episodes it gets the big treasure, but some episodes the wind pushes it around and things go wrong.

An RL agent doesn't just need to maximize expected return — in practice, stability matters too. Let's quantify this.

**Task**: For each strategy, run 2000 episodes on the noisy env and compute:
1. The probability of reaching the small treasure (position 0)
2. The probability of reaching the big treasure (position 6)
3. The probability of timing out

**Hint:** check `env.pos` after each episode ends to see where the agent ended up.

In [ ]:
# YOUR CODE HERE
# Analyze the return distributions from Exercise 4
#
# For each strategy, run episodes again but this time track
# HOW each episode ended (which treasure or timeout).
# Compute percentages for each outcome.
#
# Hint: you can check env.pos after the episode ends to see where the agent ended up.

n_episodes = 2000
env = NoisyTreasureHunter()

# Track outcomes for 'left' strategy
left_outcomes = {'small_treasure': 0, 'big_treasure': 0, 'timeout': 0}
# Track outcomes for 'right' strategy  
right_outcomes = {'small_treasure': 0, 'big_treasure': 0, 'timeout': 0}

# YOUR CODE HERE

# --- Validation ---
# Uncomment after implementing:
#
# for name, outcomes in [('Left', left_outcomes), ('Right', right_outcomes)]:
#     total = sum(outcomes.values())
#     print(f"\n{name} strategy:")
#     for outcome, count in outcomes.items():
#         print(f"  {outcome}: {count}/{total} ({100*count/total:.1f}%)")
#
# assert sum(left_outcomes.values()) == n_episodes
# assert sum(right_outcomes.values()) == n_episodes
# print("\nPassed!")

## Part 5: Why This Matters for RL

You've now built the foundation that every RL algorithm rests on. Let's connect the dots.

### What we've established:

1. **The return measures strategy quality.** But we need discounting to handle uncertainty and infinite horizons.

2. **Gamma encodes a worldview.** Low gamma = "grab what's nearby, the future is uncertain." High gamma = "plan ahead, the big payoff is worth the wait." The right gamma depends on the environment.

3. **Returns are noisy.** A single episode doesn't tell you which strategy is better. You need many episodes — and even then, variance matters.

4. **The return at each timestep matters** — not just the total. An agent needs to know "from HERE, how much reward can I expect?" That per-timestep return is what the **value function** will learn to predict in the next notebook.

### Looking ahead:

In our Treasure Hunter, we evaluated strategies by running them many times and averaging returns. But a real agent can't do that — it needs to **estimate** how good a state is from experience it's already collected. That's what the value function does.

And the difference between "what I expected" and "what actually happened" — that's the **advantage**, which tells the agent which actions to take more or less often. That's the core of PPO.

### Exercise 6: Design Challenge

This is an open-ended exercise. No automated validation — think through it and write your reasoning.

You're designing a reward function for a drone that needs to fly from point A to point B.

**Option A**: +1 reward when it reaches B, 0 everywhere else.

**Option B**: -0.01 every step (time penalty) and +1 when it reaches B.

**Option C**: Reward = negative distance to B at each step (closer = less negative).

For each option, answer:
1. What would gamma=0 do? (What strategy would a myopic agent learn?)
2. What would gamma=0.99 do?
3. What could go wrong? (Think about unintended behaviors)
4. Which would you choose and why?

*Write your answers here:*

**Option A (sparse reward):**
- gamma=0: 
- gamma=0.99: 
- What could go wrong: 

**Option B (time penalty + sparse):**
- gamma=0: 
- gamma=0.99: 
- What could go wrong: 

**Option C (distance-based):**
- gamma=0: 
- gamma=0.99: 
- What could go wrong: 

**My choice and reasoning:**


## Summary

| Concept | What it means | Why it matters |
|---------|--------------|----------------|
| **Episode** | One complete run from start to terminal | The unit of experience |
| **Return $G_t$** | Total (discounted) reward from step $t$ onward | Measures how good a state-action sequence is |
| **Gamma $\gamma$** | Discount factor (0 to 1) | Controls how far ahead the agent plans |
| **Variance** | Returns differ across episodes | We need many episodes to evaluate a strategy |

**Key formula you'll use everywhere:**

$$G_t = r_t + \gamma \cdot G_{t+1}$$

This recursive form is the basis of TD learning, GAE, and everything in notebooks 3-4.

**Next notebook:** Policy and value functions — how a neural network learns to predict returns and choose actions.